# DistilBERT Tagalog — with Hyperparameter Tuning

Optuna search over learning rate, batch size, dropout, warmup ratio and weight
decay. The best configuration found here is baked into the Config of the main
`DistilBert_Tagalog.ipynb` for final training.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    average_precision_score
)
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset
import tempfile
import subprocess
import sys
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True
)

#  IMPORT LIBRARIES AND SETUP

In [ ]:
def install_requirements():
    required_packages = [
        'transformers[torch]',
        'datasets',
        'torch',
        'pandas',
        'scikit-learn',
        'onnx',
        'onnxruntime',
        'onnxscript',  # Added to fix ONNX export dependency issue
        'optimum[onnxruntime]',  # This enables ORTModelForSequenceClassification
        'onnxconverter-common', # Required for FP16 conversion
        'optuna', # Added for Hyperparameter Optimization
        'litert-torch',  # PyTorch -> LiteRT/TFLite (formerly 'ai-edge-torch')
        'torchao',       # PT2E dynamic-range INT8 quantization flow
    ]

    # Map pip package names to their import names where they differ
    import_name_map = {
        'scikit-learn': 'sklearn',
        'transformers[torch]': 'transformers',
        'optimum[onnxruntime]': 'optimum',
        'onnxconverter-common': 'onnxconverter_common',
        'litert-torch': 'litert_torch',
    }

    for package in required_packages:
        import_name = import_name_map.get(package, package.split('[')[0])
        try:
            __import__(import_name)
        except ImportError:
            logging.info(f"Installing {package}...")
            # Non-fatal: a single bad/renamed package must not abort the whole setup.
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            except subprocess.CalledProcessError as e:
                logging.warning(f"Failed to install {package}: {e}")

In [ ]:
logging.info("Checking and installing dependencies...")
install_requirements()

In [ ]:
try:
    from optimum.onnxruntime import ORTModelForSequenceClassification
    from optimum.onnxruntime.configuration import OptimizationConfig
    ONNX_AVAILABLE = True
except ImportError:
    logging.warning("Optimum ONNX Runtime not available, ONNX export will be limited")
    ONNX_AVAILABLE = False

In [ ]:
logging.info("All dependencies loaded successfully!")

# LOAD AND PREPARE DATASET

In [ ]:
def load_dataset(csv_path='cleaned_data.csv', label2id=None):
    logging.info(f"Loading dataset from {csv_path}...")

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Dataset file '{csv_path}' not found! Please upload the dataset.")

    df = pd.read_csv(csv_path)
    logging.info(f"Dataset loaded successfully. Shape: {df.shape}")

    # Validate required columns
    required_cols = ['text', 'label']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Clean and validate data
    df = df.dropna(subset=['text', 'label'])
    df['text'] = df['text'].astype(str)

    # Enforce Label Schema if provided
    if label2id:
        logging.info(f"Enforcing label schema: {label2id}")

        # Standardize labels to lowercase string if they are strings
        if df['label'].dtype == object:
            df['label'] = df['label'].str.lower().str.strip()

            # Filter to only keep rows with valid labels
            valid_labels = list(label2id.keys())
            initial_count = len(df)
            df = df[df['label'].isin(valid_labels)]
            logging.info(f"Filtered {initial_count - len(df)} rows not matching schema.")

            # Map to IDs
            df['label'] = df['label'].map(label2id)

        elif pd.api.types.is_integer_dtype(df['label']):
             # Verify IDs match values
             valid_ids = list(label2id.values())
             df = df[df['label'].isin(valid_ids)]

    # Ensure labels are integers
    df['label'] = df['label'].astype(int)

    logging.info(f"Dataset validation complete. Clean shape: {df.shape}")
    logging.info(f"Label distribution:\n{df['label'].value_counts().sort_index()}")

    return df

In [ ]:
def split_dataset(df, train_size=0.70, val_size=0.25, test_size=0.05, random_state=42):
    logging.info(f"Splitting dataset: {train_size*100}% Train, {val_size*100}% Val, {test_size*100}% Test...")

    if not np.isclose(train_size + val_size + test_size, 1.0):
        raise ValueError(f"Split sizes must sum to 1.0. Got {train_size + val_size + test_size}")

    # First split: Train vs (Val + Test)
    # We treat (Val + Test) as the "test" portion for the first split
    first_split_test_size = val_size + test_size

    X_train, X_temp, y_train, y_temp = train_test_split(
        df['text'].tolist(),
        df['label'].tolist(),
        train_size=train_size,
        random_state=random_state,
        stratify=df['label']
    )

    # Second split: Val vs Test
    # We split X_temp into Val and Test
    # The test_size here is relative to the size of X_temp
    relative_test_size = test_size / first_split_test_size

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=relative_test_size,
        random_state=random_state,
        stratify=y_temp
    )

    logging.info(f"Training set: {len(X_train)} samples")
    logging.info(f"Validation set: {len(X_val)} samples")
    logging.info(f"Test set: {len(X_test)} samples")

    return X_train, X_val, X_test, y_train, y_val, y_test

# TOKENIZATION AND PREPROCESSING

In [ ]:
def create_tokenized_datasets(X_train, X_val, X_test, y_train, y_val, y_test, model_name, max_length=256):
    """Tokenize the datasets using the DistilBERT Tagalog tokenizer"""
    logging.info("Loading DistilBERT Tagalog tokenizer and creating tokenized datasets...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # DistilBERT uses BERT-style tokenization
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.unk_token if tokenizer.unk_token is not None else '[PAD]'

    # Create datasets
    train_dataset = Dataset.from_dict({'text': X_train, 'labels': y_train})
    val_dataset = Dataset.from_dict({'text': X_val, 'labels': y_val})
    test_dataset = Dataset.from_dict({'text': X_test, 'labels': y_test})

    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            truncation=True,
            padding=False,  # Will be handled by data collator
            max_length=max_length
        )

    # Tokenize datasets
    train_tokenized = train_dataset.map(tokenize_function, batched=True)
    val_tokenized = val_dataset.map(tokenize_function, batched=True)
    test_tokenized = test_dataset.map(tokenize_function, batched=True)

    logging.info("Tokenization complete!")
    logging.info(f"Sample tokenized text length: {len(train_tokenized[0]['input_ids'])}")

    return train_tokenized, val_tokenized, test_tokenized, tokenizer

# MODEL TRAINING

In [ ]:
def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids

    # Handle tuple predictions (e.g. from models returning more than just logits)
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Get probabilities for AUC and hard predictions for other metrics
    # Assuming predictions are logits
    if hasattr(predictions, 'numpy'):
        logits = predictions.numpy()
    else:
        logits = predictions

    # Calculate Probabilities using Softmax
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

    # Get Hard Predictions
    preds = np.argmax(logits, axis=1)

    # Calculate Metrics
    accuracy = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')
    precision_macro = precision_score(labels, preds, average='macro', zero_division=0)
    recall_macro = recall_score(labels, preds, average='macro', zero_division=0)

    # PR AUC (Focus on positive class for binary, or handle multiclass)
    pr_auc = 0.0
    try:
        if len(np.unique(labels)) == 2:
            # Binary case: Use probability of the positive class (index 1)
            # Assuming 0=Safe, 1=NSFW
            pr_auc = average_precision_score(labels, probs[:, 1])
        else:
            pass
    except Exception:
        pass

    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'pr_auc': pr_auc
    }

def run_hpo(train_dataset, val_dataset, tokenizer, model_name, num_labels, n_trials=5):
    """Run Hyperparameter Optimization using Optuna and save logs"""
    logging.info("Initializing Hyperparameter Optimization...")
    import optuna
    from transformers.trainer_utils import BestRun
    from transformers import AutoConfig
    import os

    def objective(trial):
        # Define search space (Modified as requested)
        per_device_train_batch_size = trial.suggest_categorical("per_device_train_batch_size", [16, 32])
        dropout = trial.suggest_float("dropout", 0.1, 0.5)
        learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
        warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.2)
        weight_decay = trial.suggest_float("weight_decay", 0.0, 0.3)

        # Fixed epochs for HPO efficiency
        num_train_epochs = 10

        # Define output directory for this trial
        trial_output_dir = f"hpo_temp/trial_{trial.number}"

        # Initialize Model with suggested dropout
        config = AutoConfig.from_pretrained(
            model_name,
            num_labels=num_labels,
            problem_type="single_label_classification"
        )
        # Apply dropout to classifier
        config.seq_classif_dropout = dropout

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            config=config
        )

        # Ensure model uses the correct pad_token_id
        if tokenizer.pad_token_id is not None:
            model.config.pad_token_id = tokenizer.pad_token_id

        training_args = TrainingArguments(
            output_dir=trial_output_dir,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_steps=10,
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            greater_is_better=True,
            report_to="none",
            fp16=torch.cuda.is_available(),
            # Tuned Hyperparameters
            learning_rate=learning_rate,
            per_device_train_batch_size=per_device_train_batch_size,
            per_device_eval_batch_size=per_device_train_batch_size,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,
            # Fixed for HPO
            num_train_epochs=num_train_epochs,
            seed=42,
            # Disable full saving to save space during HPO
            save_total_limit=1,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=tokenizer,
            compute_metrics=compute_metrics,
        )

        trainer.train()

        # Return the best metric found during training
        return trainer.state.best_metric

    # Run Optimization
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    # Save Logs
    try:
        df_logs = study.trials_dataframe()

        # Save to metrics directory
        os.makedirs('metrics', exist_ok=True)
        log_path = "metrics/hpo_trials_explored.csv"
        df_logs.to_csv(log_path, index=False)
        logging.info(f"HPO trials/hyperparameters list saved to {log_path}")
    except Exception as e:
        logging.warning(f"Failed to save HPO logs: {e}")

    # Return Best Run
    best_trial = study.best_trial
    return BestRun(str(best_trial.number), best_trial.value, best_trial.params)

def train_model(train_dataset, val_dataset, tokenizer, model_name, output_dir, num_labels, config):
    """Train the DistilBERT Tagalog model for classification"""
    logging.info("Initializing DistilBERT Tagalog model for classification training...")
    from transformers import AutoConfig

    # Load model config and apply hyperparameters
    model_config = AutoConfig.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type="single_label_classification"
    )

    # Apply Dropout from Config
    if hasattr(config, 'DROPOUT'):
        logging.info(f"Applying Dropout: {config.DROPOUT}")
        model_config.seq_classif_dropout = config.DROPOUT

    # Load model with updated config
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=model_config
    )

    # Ensure model uses the correct pad_token_id
    if tokenizer.pad_token_id is not None:
        model.config.pad_token_id = tokenizer.pad_token_id

    # Data collator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Check for GPU
    use_fp16 = torch.cuda.is_available()
    if use_fp16:
        logging.info(f"GPU detected: {torch.cuda.get_device_name(0)}")
    else:
        logging.info("No GPU detected. Training will be slower.")

    # Training arguments optimized for DistilBERT Tagalog on GPU
    training_args_dict = {
        'output_dir': output_dir,
        'num_train_epochs': config.NUM_EPOCHS,
        'per_device_train_batch_size': config.BATCH_SIZE,
        'per_device_eval_batch_size': config.BATCH_SIZE,
        'learning_rate': config.LEARNING_RATE,
        'weight_decay': config.WEIGHT_DECAY,
        'warmup_ratio': config.WARMUP_RATIO,
        'logging_dir': f'{output_dir}/logs',
        'logging_steps': config.LOGGING_STEPS,
        'save_total_limit': config.SAVE_TOTAL_LIMIT,
        'load_best_model_at_end': True,
        'metric_for_best_model': "eval_f1_macro",
        'greater_is_better': True,
        'report_to': [],
        'seed': 42,
        'dataloader_num_workers': 2,
        'remove_unused_columns': True,
        'fp16': use_fp16,
        'dataloader_pin_memory': True,
        'gradient_checkpointing': False,
    }

    # Add version-specific parameters
    if hasattr(TrainingArguments, 'eval_strategy'):
        training_args_dict['eval_strategy'] = "epoch"
        training_args_dict['save_strategy'] = "epoch"
    else:
        training_args_dict['evaluation_strategy'] = "epoch"
        training_args_dict['save_strategy'] = "epoch"

    training_args = TrainingArguments(**training_args_dict)

    # Initialize trainer using the global compute_metrics
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    logging.info("Starting training...")
    trainer.train()

    # Save training logs
    try:
        log_history = trainer.state.log_history
        if log_history:
            df_logs = pd.DataFrame(log_history)
            log_path = os.path.join(output_dir, "training_logs.csv")
            os.makedirs(output_dir, exist_ok=True)
            df_logs.to_csv(log_path, index=False)
            logging.info(f"Training logs saved to {log_path}")
    except Exception as e:
        logging.warning(f"Failed to save training logs: {e}")

    logging.info(f"Saving model to {output_dir}...")
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)

    return trainer, model

# MODEL EVALUATION

In [ ]:
def evaluate_model(trainer, tokenized_dataset, output_dir, num_labels, id2label=None, stage="validation", threshold=None):
    logging.info(f"Evaluating model performance on {stage} set...\n(Threshold: {threshold if threshold else 'Default/Argmax'})")
    import torch
    import json
    import pandas as pd

    # Get predictions and runtime metrics on the specific dataset
    eval_results = trainer.evaluate(eval_dataset=tokenized_dataset)

    # Get detailed predictions for classification report
    predictions = trainer.predict(tokenized_dataset)

    # Extract predictions from the prediction object
    if hasattr(predictions, 'predictions'):
        preds = predictions.predictions
    else:
        preds = predictions[0]

    # Handle tuple output
    if isinstance(preds, tuple):
        preds = preds[0]

    # Convert to numpy array
    if hasattr(preds, 'numpy'):
        logits = preds.numpy()
    elif torch.is_tensor(preds):
        logits = preds.cpu().numpy()
    else:
        logits = preds

    # Calculate Probabilities
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

    # Hard Predictions
    if threshold is not None and num_labels == 2:
        # Binary classification thresholding
        # Assuming index 1 is the positive class (e.g., NSFW)
        y_pred = (probs[:, 1] >= threshold).astype(int)
    else:
        # Default Argmax
        y_pred = np.argmax(probs, axis=1)

    # Extract Ground Truth Labels from dataset
    y_true = tokenized_dataset['labels']

    # Determine target names
    target_names = None
    if id2label:
        target_names = [id2label[i] for i in range(num_labels)]
    else:
        target_names = [f'Class_{i}' for i in range(num_labels)]

    report = classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        digits=4
    )

    # Calculate additional metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)

    # PR-AUC
    pr_auc = 0.0
    try:
        if num_labels == 2:
            pr_auc = average_precision_score(y_true, probs[:, 1])
    except Exception as e:
        logging.warning(f"Could not calculate PR-AUC: {e}")

    # Extract Speed Metrics
    runtime = eval_results.get("eval_runtime", 0)
    samples_per_second = eval_results.get("eval_samples_per_second", 0)
    latency_ms = (runtime / len(y_true) * 1000) if len(y_true) > 0 else 0

    # --- SAVE LOGS (Detailed Predictions) ---
    try:
        # Extract text if available
        texts = tokenized_dataset['text'] if 'text' in tokenized_dataset.column_names else range(len(y_true))

        # Create DataFrame
        df_log = pd.DataFrame({
            'text': texts,
            'true_label': y_true,
            'predicted_label': y_pred,
            'confidence': np.max(probs, axis=1)
        })

        # Map labels to strings if provided
        if id2label:
            df_log['true_label_str'] = df_log['true_label'].map(id2label)
            df_log['predicted_label_str'] = df_log['predicted_label'].map(id2label)

        # Save to CSV
        os.makedirs('metrics', exist_ok=True)
        log_path = f'metrics/distilbert_tagalog_{stage.lower()}_logs.csv'
        df_log.to_csv(log_path, index=False)
        logging.info(f"Detailed {stage} logs saved to {log_path}")

    except Exception as e:
        logging.warning(f"Failed to save {stage} logs: {e}")
    # ----------------------------------------

    # Prepare metrics text
    metrics_text = f"""DistilBERT Tagalog Classification Model Evaluation Results ({stage})
{'='*70}

Model: jcblaise/distilbert-tagalog-base-cased
Task: Tagalog Text Classification ({num_labels} classes)
Schema: {id2label if id2label else 'Auto'}
Threshold: {threshold if threshold else 'Default (Argmax)'}

Performance Metrics:
{'-'*30}
Accuracy: {accuracy:.4f}
Precision (Macro): {precision_macro:.4f}
Recall (Macro): {recall_macro:.4f}
F1-Score (Macro): {f1_macro:.4f}
F1-Score (Weighted): {f1_weighted:.4f}
PR-AUC (Binary/Positive): {pr_auc:.4f}

Inference Speed Metrics ({stage} Set):
{'-'*30}
Total Runtime: {runtime:.4f} seconds
Samples per Second (FPS): {samples_per_second:.2f}
Latency per Sample: {latency_ms:.4f} ms

Classification Report:
{report}

Training Results:
{'-'*30}
"""

    for key, value in eval_results.items():
        if isinstance(value, (int, float)):
            metrics_text += f"{key}: {value:.4f}\n"

    # Save metrics
    metrics_path = f'metrics/distilbert_tagalog_{stage.lower()}_metrics.txt'
    with open(metrics_path, 'w') as f:
        f.write(metrics_text)
    logging.info(f"Metrics for {stage} saved to {metrics_path}")

    # Save dedicated classification report
    report_path = f'metrics/classification_report_{stage.lower()}.txt'
    with open(report_path, 'w') as f:
        f.write(report)
    logging.info(f"Classification report for {stage} saved to {report_path}")

    logging.info(f"{stage} Accuracy: {accuracy:.4f} | F1-Macro: {f1_macro:.4f}")

    return accuracy, report

def find_optimal_threshold(trainer, tokenized_dataset, num_labels, id2label=None):
    """Finds the optimal probability threshold to maximize F1-Macro on the validation set."""
    logging.info("\n--- Tuning Classification Threshold ---")

    if num_labels != 2:
        logging.info("Multiclass classification detected. Skipping threshold tuning (using Argmax).")
        return None

    # Get raw probabilities
    predictions = trainer.predict(tokenized_dataset)
    preds = predictions.predictions
    if isinstance(preds, tuple): preds = preds[0]
    logits = preds.numpy() if hasattr(preds, 'numpy') else preds
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
    y_true = tokenized_dataset['labels']

    # Search a sane band, only deviating from the neutral 0.5 if the validation
    # gain is large enough to be real rather than noise. This mirrors the
    # RoBERTa / LSTM / BiLSTM threshold policy so all four models share an
    # operating point and stay directly comparable.
    thresholds = np.arange(0.30, 0.71, 0.01)
    f1_scores = [f1_score(y_true, (probs[:, 1] >= t).astype(int), average='macro') for t in thresholds]
    best_idx = int(np.argmax(f1_scores))
    tuned_thresh = float(thresholds[best_idx])
    tuned_f1 = f1_scores[best_idx]

    f1_at_half = f1_score(y_true, (probs[:, 1] >= 0.5).astype(int), average='macro')
    if tuned_f1 - f1_at_half > 0.01:
        best_thresh, best_f1 = tuned_thresh, tuned_f1
    else:
        best_thresh, best_f1 = 0.5, f1_at_half

    logging.info(f"Optimal Threshold: {best_thresh:.2f} (tuned {tuned_thresh:.2f}->F1 {tuned_f1:.4f} vs 0.50->F1 {f1_at_half:.4f})")
    logging.info(f"Best Validation F1-Macro: {best_f1:.4f}")
    return best_thresh

# ONNX EXPORT

In [ ]:
def export_to_onnx(model_dir, onnx_path, max_length=256):
    logging.info("Exporting DistilBERT Tagalog model to ONNX format...")

    try:
        # Load the trained model
        model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        tokenizer = AutoTokenizer.from_pretrained(model_dir)

        # Ensure model is on CPU for export
        model.cpu()
        model.eval()

        # Create dummy input with Tagalog sample
        dummy_input = tokenizer(
            "Ito ay isang sample na Tagalog text para sa ONNX export",
            return_tensors="pt",
            max_length=max_length,
            padding="max_length",
            truncation=True
        )

        # Define paths
        onnx_path_fp32 = onnx_path.replace(".onnx", "_fp32.onnx")

        # Export to ONNX (Float32 first)
        os.makedirs(os.path.dirname(onnx_path), exist_ok=True)

        # Explicitly unpack inputs for clarity and safety
        input_ids = dummy_input['input_ids']
        attention_mask = dummy_input['attention_mask']

        torch.onnx.export(
            model,
            (input_ids, attention_mask),
            onnx_path_fp32,
            export_params=True,
            opset_version=14,
            do_constant_folding=True,
            input_names=['input_ids', 'attention_mask'],
            output_names=['logits'],
            dynamic_axes={
                'input_ids': {0: 'batch_size', 1: 'sequence'},
                'attention_mask': {0: 'batch_size', 1: 'sequence'},
                'logits': {0: 'batch_size'}
            }
        )

        logging.info(f"Intermediate ONNX (Float32) exported to: {onnx_path_fp32}")

        # Convert to FP16
        logging.info("Converting ONNX model to FP16...")
        import onnx
        from onnxconverter_common import float16

        model_fp32 = onnx.load(onnx_path_fp32)
        model_fp16 = float16.convert_float_to_float16(model_fp32)

        # Save FP16 model to the target path
        onnx.save(model_fp16, onnx_path)
        logging.info(f"ONNX FP16 model exported to: {onnx_path}")

        # Get model size
        model_size = os.path.getsize(onnx_path) / (1024 * 1024)
        logging.info(f"ONNX FP16 model size: {model_size:.2f} MB")

        return True

    except Exception as e:
        logging.error(f"ONNX export failed: {e}")
        import traceback
        traceback.print_exc()
        return False

In [ ]:
def validate_onnx_model(onnx_path, tokenizer, max_length=256):
    """Validates the exported ONNX model using ONNX Runtime"""
    logging.info(f"\nValidating ONNX model with ONNX Runtime: {onnx_path}")
    import onnxruntime as ort
    import numpy as np
    import os

    if not os.path.exists(onnx_path):
        logging.error(f"❌ Error: Model file not found at {onnx_path}")
        return False

    try:
        # Load ONNX model
        ort_session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
        logging.info("✅ ONNX InferenceSession created successfully")

        # Create dummy input
        text = "Subukan natin kung gumagana ang ONNX model."
        inputs = tokenizer(
            text,
            return_tensors="np",
            padding="max_length",
            max_length=max_length,
            truncation=True
        )

        # Prepare inputs for ORT
        ort_inputs = {
            'input_ids': inputs['input_ids'].astype(np.int64),
            'attention_mask': inputs['attention_mask'].astype(np.int64)
        }

        # Run inference
        ort_outs = ort_session.run(None, ort_inputs)
        logits = ort_outs[0]

        # Calculate probabilities (Softmax)
        def softmax(x):
            e_x = np.exp(x - np.max(x))
            return e_x / e_x.sum(axis=1, keepdims=True)

        probs = softmax(logits)
        pred_label = np.argmax(probs, axis=1)[0]

        logging.info(f"Input: '{text}'")
        logging.info(f"Predicted Class Index: {pred_label}")
        logging.info(f"Confidence: {probs[0][pred_label]:.4f}")
        logging.info("✅ ONNX Runtime validation passed!")
        return True

    except Exception as e:
        logging.error(f"❌ ONNX Validation Failed: {e}")
        import traceback
        traceback.print_exc()
        return False

# TENSORFLOW LITE EXPORT

In [ ]:
def export_to_tflite_ai_edge(model_dir, output_dir, max_length=256):
    """Convert the fine-tuned PyTorch model to TFLite via litert-torch (LiteRT).

    transformers v5 dropped native TF, so the old TFAutoModel(from_pt=True) ->
    TFLite route no longer works. litert-torch (formerly 'ai-edge-torch')
    converts PyTorch -> LiteRT directly and supports a data-free dynamic-range
    INT8 recipe. This matches the DOST_RoBERTa export path so both transformer
    models ship comparable artifacts.

    Emits:
      - distilbert_tagalog_float32.tflite : reference build (largest)
      - distilbert_tagalog_int8.tflite    : dynamic-range INT8 weights (~1/4 size)
    Both import via Android Studio's ML Model Binding.
    """
    import importlib

    # 'ai-edge-torch' was renamed to 'litert-torch'; the old name is now a stub
    # without .convert. Use whichever module actually provides the API.
    aet = None
    for modname in ('litert_torch', 'ai_edge_torch'):
        try:
            m = importlib.import_module(modname)
        except Exception:
            continue
        if hasattr(m, 'convert'):
            aet = m
            break
    if aet is None:
        logging.error("No working litert_torch/ai_edge_torch (.convert) found; skipping TFLite export.")
        return False
    pkg = aet.__name__
    logging.info(f"Using '{pkg}' for TFLite conversion.")

    try:
        # Reload a clean CPU eval model. eager attention avoids SDPA's
        # data-dependent branches, which torch.export cannot trace.
        model = AutoModelForSequenceClassification.from_pretrained(
            model_dir, attn_implementation="eager"
        ).eval().to("cpu")
        tokenizer = AutoTokenizer.from_pretrained(model_dir)

        class _LogitsWrapper(torch.nn.Module):
            """HF returns a dataclass; expose just the logits tensor for tracing."""
            def __init__(self, m):
                super().__init__()
                self.m = m

            def forward(self, input_ids, attention_mask):
                return self.m(input_ids=input_ids, attention_mask=attention_mask).logits

        wrapper = _LogitsWrapper(model).eval()

        # Fixed [1, max_length] sample -> the .tflite takes a fixed-length input.
        enc = tokenizer(
            "Ito ay isang sample na Tagalog text", return_tensors="pt",
            max_length=max_length, padding="max_length", truncation=True
        )
        sample = (enc["input_ids"], enc["attention_mask"])

        os.makedirs(output_dir, exist_ok=True)
        ok = False

        # --- Float32 LiteRT model (reliable baseline) ---
        try:
            logging.info(f"Converting PyTorch -> TFLite (float32) via {pkg}...")
            edge = aet.convert(wrapper, sample)
            f32 = os.path.join(output_dir, "distilbert_tagalog_float32.tflite")
            edge.export(f32)
            logging.info(f"TFLite saved to {f32} ({os.path.getsize(f32) / (1024 * 1024):.2f} MB)")
            ok = True
        except Exception as e:
            logging.error(f"{pkg} float32 conversion failed: {e}")
            import traceback
            traceback.print_exc()

        # --- Dynamic-range INT8 (data-free; ~1/4 size) ---
        try:
            pt2e = importlib.import_module(f"{pkg}.quantize.pt2e_quantizer")
            qcfg = importlib.import_module(f"{pkg}.quantize.quant_config")
            PT2EQuantizer = pt2e.PT2EQuantizer
            get_symmetric_quantization_config = pt2e.get_symmetric_quantization_config
            QuantConfig = qcfg.QuantConfig

            # prepare_pt2e/convert_pt2e moved out of torch.ao into torchao in newer torch.
            prepare_pt2e = convert_pt2e = None
            for modpath in (
                'torchao.quantization.pt2e.quantize_pt2e',
                'torch.ao.quantization.quantize_pt2e',
                'torch.ao.quantization.pt2e.quantize_pt2e',
            ):
                try:
                    qmod = importlib.import_module(modpath)
                    prepare_pt2e, convert_pt2e = qmod.prepare_pt2e, qmod.convert_pt2e
                    break
                except Exception:
                    continue
            if prepare_pt2e is None:
                raise ImportError("prepare_pt2e/convert_pt2e not found (torch.ao or torchao)")

            logging.info(f"Converting PyTorch -> TFLite (dynamic-range INT8) via {pkg} PT2E...")
            quantizer = PT2EQuantizer().set_global(
                get_symmetric_quantization_config(is_per_channel=True, is_dynamic=True)
            )

            # Capture a pre-autograd graph (entry point name varies across torch versions).
            captured = None
            try:
                captured = torch.export.export_for_training(wrapper, sample).module()
            except (AttributeError, TypeError):
                try:
                    from torch._export import capture_pre_autograd_graph
                    captured = capture_pre_autograd_graph(wrapper, sample)
                except Exception:
                    captured = torch.export.export(wrapper, sample).module()

            prepared = prepare_pt2e(captured, quantizer)
            converted = convert_pt2e(prepared)
            edge_q = aet.convert(
                converted, sample, quant_config=QuantConfig(pt2e_quantizer=quantizer)
            )
            i8 = os.path.join(output_dir, "distilbert_tagalog_int8.tflite")
            edge_q.export(i8)
            logging.info(f"TFLite saved to {i8} ({os.path.getsize(i8) / (1024 * 1024):.2f} MB)")
            ok = True
        except Exception as e:
            logging.error(f"{pkg} INT8 conversion failed: {e}")
            import traceback
            traceback.print_exc()

        if ok:
            logging.info("Import in Android Studio via: File > New > Other > TensorFlow Lite Model.")
        else:
            logging.error(f"{pkg} produced no TFLite files.")
        return ok

    except Exception as e:
        logging.error(f"TFLite export failed: {e}")
        import traceback
        traceback.print_exc()
        return False

# INFERENCE

In [ ]:
def test_inference(model_dir, max_length=256, test_texts=None, id2label=None):
    if test_texts is None:
        # Default texts tailored for Safe/NSFW checks
        test_texts = [
            "Magandang umaga po sa inyong lahat!",  # Safe
            "Maraming salamat sa tulong mo.",       # Safe
            "Putang ina mo gago ka!",               # NSFW (Profanity)
            "Walang kwenta kausap, bobo.",          # NSFW (Insult)
        ]

    logging.info("\nTesting trained DistilBERT Tagalog model...")

    results = []

    try:
        model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        tokenizer = AutoTokenizer.from_pretrained(model_dir)

        model.eval()

        for i, text in enumerate(test_texts):
            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=max_length,
                padding=True
            )

            with torch.no_grad():
                outputs = model(**inputs)
                predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
                predicted_class = torch.argmax(predictions, dim=-1).item()
                confidence = predictions[0][predicted_class].item()

            label_str = f"Class {predicted_class}"
            if id2label and predicted_class in id2label:
                label_str = id2label[predicted_class].upper()

            logging.info(f"Text {i+1}: '{text}' -> {label_str} ({confidence:.4f})")

            # Collect result
            results.append({
                'text': text,
                'predicted_label_id': predicted_class,
                'predicted_label': label_str,
                'confidence': confidence
            })

        # Save results to CSV
        if results:
            import pandas as pd
            df_inference = pd.DataFrame(results)
            os.makedirs('metrics', exist_ok=True)
            log_path = 'metrics/distilbert_tagalog_inference_logs.csv'
            df_inference.to_csv(log_path, index=False)
            logging.info(f"Inference logs saved to {log_path}")

    except Exception as e:
        logging.error(f"Inference test failed: {e}")
        import traceback
        traceback.print_exc()

# MAIN

In [ ]:
class Config:
    # Model & Paths
    MODEL_NAME = "jcblaise/distilbert-tagalog-base-cased"
    DATASET_PATH = 'cleaned_data.csv'
    OUTPUT_DIR = "models/distilbert_tagalog_classification"
    ONNX_PATH = "models/distilbert_tagalog_classification_model.onnx"
    TFLITE_PATH = "models/distilbert_tagalog_classification_model.tflite"

    # Label Schema (Enforced)
    LABEL2ID = {"safe": 0, "nsfw": 1}
    ID2LABEL = {0: "safe", 1: "nsfw"}
    NUM_LABELS = 2

    # Data Processing - Split 70/15/15 (aligned with RoBERTa / LSTM / BiLSTM)
    TRAIN_SIZE = 0.70
    VAL_SIZE = 0.15
    TEST_SIZE = 0.15
    RANDOM_STATE = 42
    MAX_LENGTH = 256

    # Training Hyperparameters
    NUM_EPOCHS = 50
    BATCH_SIZE = 32
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    DROPOUT = 0.1 # Added for tuning
    LOGGING_STEPS = 10
    SAVE_TOTAL_LIMIT = 2

logging.info("Configuration loaded.")

In [ ]:
# Step 1: Load and Split Dataset
df = load_dataset(Config.DATASET_PATH, label2id=Config.LABEL2ID)
num_labels = Config.NUM_LABELS

# Split Dataset (70/25/5)
X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
    df,
    train_size=Config.TRAIN_SIZE,
    val_size=Config.VAL_SIZE,
    test_size=Config.TEST_SIZE,
    random_state=Config.RANDOM_STATE
)

In [ ]:
# Step 2: Tokenization
train_dataset, val_dataset, test_dataset, tokenizer = create_tokenized_datasets(
    X_train, X_val, X_test, y_train, y_val, y_test,
    Config.MODEL_NAME,
    max_length=Config.MAX_LENGTH
)

In [ ]:
# Step 3: Hyperparameter Optimization & Model Training
import json

# 1. Run Hyperparameter Optimization
logging.info("Starting Hyperparameter Optimization (HPO)...")
best_run = run_hpo(
    train_dataset,
    val_dataset,
    tokenizer,
    Config.MODEL_NAME,
    num_labels=Config.NUM_LABELS,
    n_trials=5 # Adjust number of trials as needed
)

logging.info(f"Best HPO Run: {best_run}")

# 2. Save HPO Results
hpo_results_path = os.path.join(Config.OUTPUT_DIR, "hpo_results.json")
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
with open(hpo_results_path, "w") as f:
    json.dump(best_run.hyperparameters, f, indent=4)
logging.info(f"HPO Results saved to {hpo_results_path}")

# 3. Apply Best Hyperparameters to Config
logging.info("Applying best hyperparameters for final training...")
Config.LEARNING_RATE = best_run.hyperparameters.get('learning_rate', Config.LEARNING_RATE)
Config.BATCH_SIZE = best_run.hyperparameters.get('per_device_train_batch_size', Config.BATCH_SIZE)
Config.DROPOUT = best_run.hyperparameters.get('dropout', Config.DROPOUT)
Config.WARMUP_RATIO = best_run.hyperparameters.get('warmup_ratio', Config.WARMUP_RATIO)
Config.WEIGHT_DECAY = best_run.hyperparameters.get('weight_decay', Config.WEIGHT_DECAY)

# Note: NUM_EPOCHS is kept at the Config default (50) for final training,
# while HPO used a smaller fixed number for speed.

# 4. Train Model with Optimized Hyperparameters
trainer, model = train_model(
    train_dataset, val_dataset, tokenizer,
    Config.MODEL_NAME,
    Config.OUTPUT_DIR,
    num_labels,
    config=Config  # Passing updated class
)

In [ ]:
# Step 4: Evaluate Model

# 1. Validate on Validation Set (Baseline / Default Threshold)
logging.info("\n--- Validation Set Evaluation (Baseline) ---")
val_accuracy, val_report = evaluate_model(
    trainer,
    val_dataset,
    Config.OUTPUT_DIR,
    num_labels,
    id2label=Config.ID2LABEL,
    stage="Validation_Baseline"
)

# 2. Tune Threshold on Validation Set
optimal_threshold = find_optimal_threshold(
    trainer,
    val_dataset,
    num_labels,
    id2label=Config.ID2LABEL
)

# 3. Evaluate on Test Set (Using Optimal Threshold)
# The "Freeze model" step is implicit as we are using the trained `trainer` without calling .train() again.
logging.info("\n--- Test Set Evaluation (Final) ---")
test_accuracy, test_report = evaluate_model(
    trainer,
    test_dataset,
    Config.OUTPUT_DIR,
    num_labels,
    id2label=Config.ID2LABEL,
    stage="Test",
    threshold=optimal_threshold  # Apply the tuned threshold here
)

logging.info(f"Final Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Step 5: Export Models (ONNX & TFLite)
onnx_success = export_to_onnx(Config.OUTPUT_DIR, Config.ONNX_PATH, max_length=Config.MAX_LENGTH)
tflite_success = export_to_tflite_ai_edge(Config.OUTPUT_DIR, Config.OUTPUT_DIR, max_length=Config.MAX_LENGTH)

if onnx_success:
    logging.info(f"✅ ONNX model exported: {Config.ONNX_PATH}")
else:
    logging.error("❌ ONNX export failed")

if tflite_success:
    logging.info(f"✅ TFLite model exported: {Config.TFLITE_PATH}")
else:
    logging.error("❌ TFLite export failed")

In [ ]:
# Step 5.1: Validate ONNX Model
if os.path.exists(Config.ONNX_PATH):
    validate_onnx_model(Config.ONNX_PATH, tokenizer, max_length=Config.MAX_LENGTH)
else:
    logging.warning("ONNX model not found. Please ensure the export step was successful.")

In [ ]:
# Step 6: Test Inference
# Using samples from the Test partition (X_test)
logging.info(f"Running inference on 10 samples from the Test Set ({len(X_test)} total)...")

# Select first 10 samples from X_test
test_sample_texts = X_test[:10]

test_inference(
    Config.OUTPUT_DIR,
    max_length=Config.MAX_LENGTH,
    test_texts=test_sample_texts,
    id2label=Config.ID2LABEL
)

In [ ]:
import os
import subprocess
import logging

def download_outputs():
    logging.info("Zipping and downloading all output files...")

    output_filename = 'distilbert_tagalog_output.zip'

    # Command to zip everything in current dir, excluding sample_data, input csv, and temporary HPO/system files
    # We exclude hpo_temp (checkpoints) to save space and time
    cmd = f'zip -r {output_filename} . -x "sample_data*" "cleaned_data.csv" ".config*" "{output_filename}" "__pycache__*" "hpo_temp*" "runs*"'

    try:
        logging.info(f"Executing: {cmd}")
        subprocess.check_call(cmd, shell=True)

        if os.path.exists(output_filename):
            file_size = os.path.getsize(output_filename) / (1024 * 1024)
            logging.info(f"Zip file created: {file_size:.2f} MB")
            logging.info(f"Downloading {output_filename}...")
            from google.colab import files
            files.download(output_filename)
        else:
            logging.error("Error: Zip file was not created.")

    except subprocess.CalledProcessError as e:
        logging.error(f"Zipping failed: {e}")
    except ImportError:
        logging.warning("Not running in Google Colab, cannot download.")
    except Exception as e:
        logging.error(f"An error occurred: {e}")

download_outputs()